# Principle-Guided Attention (PGA) vs Baseline MicroGPT

This notebook implements and benchmarks the **Principle-Guided Attention (PGA)** architecture against a standard **MicroGPT** baseline. 

### Key Features:
- **Vectorized PGA**: Uses `torch.linalg.svd` with sliding window views (`unfold`) to compute principle components for the entire sequence in parallel on the GPU.
- **Shakespeare Dataset**: Trains on the classic tiny shakespeare dataset.
- **Comparison**: Runs both models side-by-side for a specified number of steps and plots the training loss.
- **Robustness Fix**: The Principle Matrix computation (SVD) is detached from the gradient graph to prevent numerical instability (`NaN` losses) commonly associated with differencing through singular values.


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import matplotlib.pyplot as plt
import os
import requests

# Check for GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

Using device: cpu


## 1. Data Loading (Tiny Shakespeare)

In [11]:
# Download the tiny shakespeare dataset
file_path = 'input.txt'
if not os.path.exists(file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(file_path, 'w') as f:
        f.write(requests.get(data_url).text)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

# distinct characters
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Dataset stats: {len(text)} chars, {vocab_size} unique chars.")

# Simple Tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train/Val Split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split, batch_size, block_size):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Dataset stats: 1115394 chars, 65 unique chars.


## 2. Configuration

In [12]:
# Hyperparameters
BATCH_SIZE = 32
BLOCK_SIZE = 64  # Context window
MAX_ITERS = 2000 # Total training steps
EVAL_INTERVAL = 250
LEARNING_RATE = 3e-4
N_EMBD = 64      # Embedding dimension
N_HEAD = 4
N_LAYER = 4
DROPOUT = 0.1
TARGET_RANK = 8  # For PGA truncation (approx N_EMBD / 8 or 4?)
                 # Usually d_model/2 or sqrt(d_model). Let's use 8.

print(f"Config: Block={BLOCK_SIZE}, Embd={N_EMBD}, Heads={N_HEAD}, Layers={N_LAYER}")

Config: Block=64, Embd=64, Heads=4, Layers=4


## 3. Reference Modules (Norm, FeedForward)

In [13]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        ms = (x ** 2).mean(dim=-1, keepdim=True)
        scale = (ms + self.eps).rsqrt()
        return self.weight * x * scale

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(DROPOUT),
        )
    def forward(self, x):
        return self.net(x)

## 4. Baseline Attention

In [14]:
class BaselineHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, H)
        q = self.query(x) # (B, T, H)
        # compute affinities
        wei = q @ k.transpose(-2, -1) * (C**-0.5) # (B, T, H) @ (B, H, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # perform weighted aggregation
        v = self.value(x)
        out = wei @ v
        return out

class BaselineMultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([BaselineHead(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class BaselineBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = BaselineMultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BaselineMicroGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[BaselineBlock(N_EMBD, N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = RMSNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 5. Principle-Guided Attention (PGA) Implementation

This uses `tensor.unfold` to create sliding windows of the input sequence, then effectively batched SVD to compute the Principle Matrix $P$ for every time step $t$ in parallel.

**Note:** To ensure causal masking correctness in PGA, the "Observation Stack" for token $t$ consists of tokens $[t-K+1, \dots, t]$. We pad the start of the sequence so that even the first tokens have a valid (though zero-padded) window.

In [15]:
class PGA_MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size
        
        self.key = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.query = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.value = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)
        
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))

    def compute_principle_matrices(self, x):
        # x: (B, T, C)
        B, T, C = x.shape
        
        # 1. Windowing
        # We want a window of size K=BLOCK_SIZE (or smaller) for each T.
        # For causal consistency, window at t should be x[t-K+1 : t+1].
        K = min(T, 16) # Use a shorter window for SVD speed if needed, or T.
                       # Let's use K=16 to be consistent with 'Micro' idea, or full T.
                       # Using full BLOCK_SIZE as window.
        window_len = K
        
        # Pad left with zeros to handle start of sequence
        # Padding shape: (B, K-1, C)
        padding = torch.zeros(B, window_len - 1, C, device=x.device)
        x_padded = torch.cat([padding, x], dim=1) # (B, T+K-1, C)
        
        # Unfold to get windows
        # dimension 1, size window_len, step 1
        # Result shape: (B, num_windows, C, window_len)
        # num_windows = (T+K-1) - K + 1 = T. Correct.
        windows = x_padded.unfold(1, window_len, 1)
        
        # Permute for SVD: (B, T, window_len, C)
        windows = windows.permute(0, 1, 3, 2)
        
        # 2. Batched SVD
        # windows: (B, T, K, C)
        # P = V^T V. 
        # P MUST be detached to avoid NaN gradients from SVD.
        
        try:
            # We assume SVD convergence is usually okay for small matrices, but backprop is fatal.
            U, S, Vh = torch.linalg.svd(windows, full_matrices=False)
        except:
            # Fallback identity if SVD fails
            return torch.eye(C, device=x.device).view(1, 1, C, C).repeat(B, T, 1, 1)
            
        # 3. Truncation
        # Select top R components.
        target_rank = 8 # defined in config
        
        # Vh is (B, T, R_actual, C)
        # We want top 'target_rank' rows of Vh.
        r = min(target_rank, Vh.shape[-2])
        
        V_top = Vh[..., :r, :] # (B, T, r, C)
        
        # 4. Projection Matrix P = V^T V
        # V_top transpose: (B, T, C, r)
        # (B, T, C, r) @ (B, T, r, C) -> (B, T, C, C)
        P = V_top.transpose(-2, -1) @ V_top
        
        return P

    def forward(self, x):
        B, T, C = x.shape
        
        # Standard Q, K, V computation
        k = self.key(x)   # (B, T, C)
        q = self.query(x) # (B, T, C)
        v = self.value(x) # (B, T, C)
        
        # --- PGA INTERVENTION ---
        # Compute P for every token t
        # Shape: (B, T, C, C)
        # DETACHING P TO PREVENT GRADIENT NA-N
        P = self.compute_principle_matrices(x).detach()
        
        # Apply P to Q, K, V
        # q_proj = q @ P. detach P -> P is efficient constant.
        q_proj = torch.einsum('btc,btcd->btd', q, P)
        k_proj = torch.einsum('btc,btcd->btd', k, P)
        v_proj = torch.einsum('btc,btcd->btd', v, P)
        
        # --- END PGA INTERVENTION ---
        
        # Reshape for Multi-Head Attention
        # (B, T, C) -> (B, T, n_head, head_size) -> (B, n_head, T, head_size)
        k = k_proj.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        q = q_proj.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v_proj.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        
        # Causal Attention
        wei = q @ k.transpose(-2, -1) * (self.head_size**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        out = wei @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class PGABlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = PGA_MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class PGAMicroGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[PGABlock(N_EMBD, N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = RMSNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            # PGA requires careful handling of context windows, 
            # but since forward() recomputes P from scratch based on provided context,
            # simple iterative generation works fine.
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 6. Training & Comparison

In [16]:
@torch.no_grad()
def estimate_loss(model):
    result = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(100)
        for k in range(100):
            X, Y = get_batch(split, BATCH_SIZE, BLOCK_SIZE)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        result[split] = losses.mean()
    model.train()
    return result

def train_model(model, name, steps):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_history = []
    
    start_time = time.time()
    print(f"--- Training {name} for {steps} steps ---")
    
    for iter in range(steps):
        if iter % EVAL_INTERVAL == 0 or iter == steps - 1:
            losses = estimate_loss(model)
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            loss_history.append(losses['val'].item())
            
        xb, yb = get_batch('train', BATCH_SIZE, BLOCK_SIZE)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    
    dt = time.time() - start_time
    print(f"{name} finished in {dt:.2f}s")
    return loss_history

### Smoke Test

In [17]:
# Quick sanity check
print("Running smoke test...")
dummy_x = torch.zeros((2, 8), dtype=torch.long, device=device)
try:
    base_model = BaselineMicroGPT().to(device)
    pga_model = PGABlock(N_EMBD, N_HEAD).to(device) # Test just the block first or full model
    pga_full = PGAMicroGPT().to(device)
    
    out_base, _ = base_model(dummy_x)
    print(f"Baseline Output Shape: {out_base.shape}")
    
    out_pga, _ = pga_full(dummy_x)
    print(f"PGA Output Shape: {out_pga.shape}")
    print("Smoke test passed!")
except Exception as e:
    print(f"Smoke test failed: {e}")
    raise e

Running smoke test...
Baseline Output Shape: torch.Size([2, 8, 65])
PGA Output Shape: torch.Size([2, 8, 65])
Smoke test passed!


### Run Experiment Loop

In [19]:
# Initialize models
baseline_model = BaselineMicroGPT().to(device)
pga_model = PGAMicroGPT().to(device)

# Experiment Steps
experiment_steps = [500, 1000, 1500, 2000]
current_step = 0

baseline_losses = []
pga_losses = []

# We will train incrementally to hit the checkpoints
for target in experiment_steps:
    delta = target - current_step
    if delta > 0:
        print(f"\nTraining to {target} steps...")
        p_hist = train_model(pga_model, "PGA", delta)
        b_hist = train_model(baseline_model, "Baseline", delta)
        
        # Store only the final validation loss at this checkpoint for plotting
        # (Or we could aggregate the whole history)
        # Let's just store the last eval loss
        baseline_losses.append(b_hist[-1])
        pga_losses.append(p_hist[-1])
        
    current_step = target

print("Experiment Complete.")


Training to 500 steps...
--- Training PGA for 500 steps ---
step 0: train loss 4.3028, val loss 4.2992
step 250: train loss 2.6377, val loss 2.6360
step 499: train loss 2.4898, val loss 2.4926
PGA finished in 1273.69s
--- Training Baseline for 500 steps ---
step 0: train loss 4.3472, val loss 4.3471
step 250: train loss 2.6552, val loss 2.6672
step 499: train loss 2.5082, val loss 2.5076
Baseline finished in 128.13s

Training to 1000 steps...
--- Training PGA for 500 steps ---
step 0: train loss 2.4836, val loss 2.4858
step 250: train loss 2.4145, val loss 2.4238
step 499: train loss 2.3737, val loss 2.3815
PGA finished in 1245.02s
--- Training Baseline for 500 steps ---
step 0: train loss 2.5102, val loss 2.5116
step 250: train loss 2.4356, val loss 2.4462
step 499: train loss 2.3885, val loss 2.4017
Baseline finished in 126.25s

Training to 1500 steps...
--- Training PGA for 500 steps ---
step 0: train loss 2.3701, val loss 2.3792
step 250: train loss 2.3347, val loss 2.3488
step 49

## 7. Analysis & Visualization

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.plot(experiment_steps, baseline_losses, label='Baseline MicroGPT', marker='o')
plt.plot(experiment_steps, pga_losses, label='PGA MicroGPT', marker='s')
plt.xlabel('Training Steps')
plt.ylabel('Validation Loss')
plt.title('PGA vs Baseline: Training Progress')
plt.legend()
plt.grid(True)
plt.show()

# Sample Generation
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- Baseline Generation ---")
print(decode(baseline_model.generate(context, max_new_tokens=200)[0].tolist()))
print("\n--- PGA Generation ---")
print(decode(pga_model.generate(context, max_new_tokens=200)[0].tolist()))